# Evocative of One Another: Dungeons & Dragons and Pathfinder RPG

*Deval Mehta | General Assembly Data Science Capstone | January 2025*

### Imports

In [ ]:
# Staple imports for data handling, analysis, and calculation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP imports
from nltk import pos_tag
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet, stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Imports for model instantiation, fitting, and scoring
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Data tranformation and preprocessing imports
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Reddit Scraper
import praw
from reddit_scraper import unified_date

# Warnings - Display warnings only once, so that we are not inundated with them
from warnings import filterwarnings
filterwarnings('once')

### Load the Data in
We only need to run this cell one time. The "new" data is current as of 2024-11-25 at 0:22-UTC. Since new data changes from minute to minute, the results here are not totally reproducible if one opts to run the following cell to recollect data.

In [ ]:
# Run these lines once to collect fresh data.
# Collected data is current as of 2024-11-25 at 00:22 UTC.
# Re-running will produce different results since new posts change by the minute.

# unified_data('DungeonsAndDragons', 'all', 1000, 1000)
# unified_data('Pathfinder_RPG', 'all', 1000, 1000)

### Inspect and Handle the Data

Read in the data from the `.csv` files generated by our script `reddit_scraper.py`. The script takes the top 1000 posts of all time and the newest 1000 posts from each of our given subreddits, concatenates the listings, and removes any duplicate entries before writing out to a `.csv` file.

In [ ]:
dnd = pd.read_csv('../data/DungeonsAndDragons.csv')
pathfinder = pd.read_csv('../data/Pathfinder_RPG.csv')

display(dnd.head())
display(pathfinder.head())

Assess the number of posts we have accessed. PRAW does not meet the full requested amount in a given pull. Some entries may also have been deleted, so we will not have a full 2000 posts from each subreddit.

In [ ]:
dnd.shape, pathfinder.shape

With 1,972 D&D posts and 1,992 Pathfinder posts, we see that our classes will be fairly balanced (nearly perfectly so). For data exploration and modeling, this means we will not have to worry about the `stratify` argument when we train-test split and we will not have to weight classes when we run our classification models.

In [ ]:
dnd.info()

In [ ]:
pathfinder.info()

Most of the D&D data has no body text and we lack a decent amount from the Pathfinder data as well. As such, we consider the titles and body text together for our analysis. Let us create a new feature which combines them.

In [ ]:
# Fill NaN values with empty strings before concatenating title and body text
dnd = dnd.fillna('')
pathfinder = pathfinder.fillna('')

# Combine title and body text into a single column — our analysis corpus
# Many posts lack body text (especially r/DungeonsAndDragons, which skews toward images)
# Concatenating title + body captures the most signal while handling nulls without data loss
dnd['all_text'] = dnd['title'] + ' ' + dnd['self_text']
pathfinder['all_text'] = pathfinder['title'] + ' ' + pathfinder['self_text']

display(dnd.head())
display(pathfinder.head())

Before proceeding, we should ensure that there are no null values among `all_text`

In [ ]:
dnd['all_text'].isnull().sum(), pathfinder['all_text'].isnull().sum()

Now we can combine our two corpora into a single corpus, which we will analyze and feed to our models. If we are successful, our new corpus will contain 3,966 documents.

In [ ]:
combined_df = pd.concat([dnd, pathfinder]).reset_index(drop = True)
combined_df.shape

In [ ]:
combined_df.columns

We need one more new column: a binarized version of `subreddit`. We will create a new column called `isDnD` which will be 1 if the post comes from r/DungeonsAndDragons and 0 if the post comes from r/Pathfinder_RPG. We maintain the column `subreddit`, rather than simply mapping it, so that we may refer back to it if necessary.

In [ ]:
combined_df['isDnD'] = (combined_df['subreddit'] == 'DungeonsAndDragons').astype(int)
combined_df.head()

With the data combined, we can isolate the text and normalize it. In particular, we convert all the text in our documents to lowercase, so that, for example, "Wisdom" and "wisdom" are not counted differently. We also strip any punctuation present by invoking the `RegexpTokenizer` on all regular expression "words" (which includes unicode letters, ideograms, digits, and underscores [according to rexegg.com](https://www.rexegg.com/regex-quickstart.php). Then, we lemmatize the data by invoking `WordNetLemmatizer` before removing any stop words. This will standardize our data and prepare it for vectorization.

We preserve the id, token, subreddit, and isDnD columns for further analysis. Since sentiment analysis reads full documents, we preserve the original DataFrame separately for that purpose.

In [ ]:
# Create a new DataFrame that will retain only the information we will pass into our models.
# Change the case of each word in each document to lower.
token_df = combined_df[['id', 'all_text', 'isDnD']].copy()

In [ ]:
# Use a raw string for the regex pattern to avoid the deprecated escape sequence
retokenizer = RegexpTokenizer(r'\w+')

# Lowercase and tokenize each document
# Lowercasing ensures "Wisdom" and "wisdom" are treated as the same token
token_df['all_text'] = [retokenizer.tokenize(doc.lower()) for doc in token_df['all_text']]

token_df.head()

### Data Exploration
Now that our data has been tokenized, let's explore the data a bit. There might be some inherent differences between the posts made in each subreddit. The three criteria we consider are the length of each document in words, the tone of posts, and the most frequent (1,3)-grams. For the final criteria, we invoke `CountVectorizer`. During modeling, we will invoke `TfidfVectorizer` instead, to mitigate the influence of words that may be common to both subreddits.

#### Title and Post Length in Words

In [ ]:
# Make a copy of the tokenized DataFrame for data analysis.
eda_df = token_df.copy()

# Create a new column called "word_count" to record the length of each document
eda_df['word_count'] = [len(doc) for doc in eda_df['all_text']]

# Check that it worked
eda_df.head()

In [ ]:
# Longest and posts, along with their subreddit.
eda_df.sort_values(by = 'word_count', ascending = False)[['all_text', 'word_count', 'isDnD']].head()

In [ ]:
# Longest and posts, along with their subreddit.
eda_df.sort_values(by = 'word_count')[['all_text', 'word_count', 'isDnD']].head()

Interesting! All 5 of the longest posts come from r/Pathfinder_RPG, while all five of the shortest come from r/DungeonsAndDragons. The overall distribution

In [ ]:
plt.figure(figsize = (12, 8))
sns.histplot(data = eda_df,
             x = 'word_count',
             bins = 50,
             hue = 'isDnD',
             palette = 'dark',
             log_scale = True)
plt.xlabel('Log(Post Length in Words)', fontdict = {'fontsize': 12})
plt.ylabel('Number of Posts', fontdict = {'fontsize': 12})
plt.legend(['r/DungeonsAndDragons', 'r/Pathfinder_RPG'], title = 'Subreddit')
plt.title('Distribution of Posts by Log(Word Count) by Subreddit',
          fontdict = {'fontsize': 18});
plt.savefig('../images/posts_by_logwords.png', dpi = 300)

The full data confirms that this is, indeed, a general trend. Pathfinder, being a much more "rule-heavy" and specific game, tends to have longer posts than Dungeons & Dragons. This bodes well for our classifiers; there should be an appreciable difference between the posts of the two subreddits, since this also implies that the Pathfinder associated documents will have more words with possibly more limited frequency.

#### Sentiment Analysis
We consider the tone of the posts using the VADER sentiment analyzer built into NLTK. We forego the option of applying a sentiment analyzer specific to Reddit for now, but this could be a potential next step.

In [ ]:
# Subset to only the columns needed for sentiment analysis
senti_df = combined_df[['id', 'all_text', 'isDnD']].copy()

# Instantiate VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()


def get_sentiment_score(doc: str) -> float:
    """
    Compute the compound VADER sentiment score for a document.

    VADER's compound score summarizes overall sentiment as a value in [-1, 1],
    where -1 is most negative and 1 is most positive. We use this as a single
    numeric feature alongside the TF-IDF vectors in our classification pipeline.

    Args:
        doc: Raw text string to score.

    Returns:
        Compound sentiment score as a float in [-1.0, 1.0].
    """
    scores = analyzer.polarity_scores(doc)
    return scores['compound']


senti_df['compound_sentiment'] = senti_df['all_text'].apply(get_sentiment_score)

senti_df.head()

We can compare the sentiment score distribution by subreddit, just as we did for the word count.

In [ ]:
plt.figure(figsize = (12, 8))
sns.histplot(data = senti_df,
             x = 'compound_sentiment',
             bins = 50,
             hue = 'isDnD',
             palette = 'dark')
plt.xlabel('Compound VADER Sentiment Score', fontdict = {'fontsize': 12})
plt.ylabel('Number of Posts', fontdict = {'fontsize': 12})
plt.legend(['r/DungeonsAndDragons', 'r/Pathfinder_RPG'], title = 'Subreddit')
plt.title('Distribution of Sentiment Scores by Subreddit',
          fontdict = {'fontsize': 18});
plt.savefig('../images/sentiment_by_subreddit.png', dpi = 300)

Evidently, Pathfinder posts are far more polarized than Dungeons & Dragons posts, according to NLTK's implementation of VADER. Again, this bodes well for our classifiers.

#### Most and Least Frequent n-grams
To consider the most and least frequent words and pairs of words, we employ `CountVectorizer`. This will remove the stop words for us, as well as assign a vector to each word in the corpus. Before we can feed it a list of stop words, we must generate a custom list of stopwords from `nltk.corpus.stopwords`. We assume that both subreddits are predominantly English here, but we could amend the list of stop words to include stop words from other languages as well.

In [ ]:
# Define a list of stopwords to be nltk.corpus.stopwords.words('english')
stop_words = stopwords.words('english')

# Remove all the links from the data (Thanks Angelo!)
combined_df['all_text'] = combined_df['all_text'].str.replace(r'http\S+', '', regex=True).str.strip()

# Instantiate CountVectorizer
countvec = CountVectorizer(stop_words = stop_words,
                           min_df = 0.01,
                           ngram_range = (1, 3))

# Fit and transform the CountVectorizer object to the reddit corpus
cvec_posts = countvec.fit_transform(combined_df['all_text'])

# Identify the features
features = countvec.get_feature_names_out()

# Save to a dataframe
vector_df = pd.DataFrame(
    cvec_posts.toarray(),
    columns = features
)

vector_df['sub'] = combined_df['subreddit']
# Check that it worked
vector_df.head()

Let us consider now the most frequent n-grams by subreddit. It could be the vernacular of each is appreciably different.

In [ ]:
# Split vector_df by subreddit
dnd_vec = vector_df[vector_df['sub'] == 'DungeonsAndDragons']
dnd_vec.head()

In [ ]:
pathfinder_vec = vector_df[vector_df['sub'] == 'Pathfinder_RPG']

In [ ]:
dnd_freqs = dnd_vec.sum(numeric_only = True).sort_values(ascending = False).head(20)
dnd_freqs.head()

In [ ]:
pathfinder_freqs = pathfinder_vec.sum(numeric_only = True).sort_values(ascending = False).head(20)

In [ ]:
ngram_freqs = pd.DataFrame([dnd_freqs, pathfinder_freqs]).T.reset_index()
ngram_freqs = ngram_freqs.rename(
    columns={'index': 'ngram', 0: 'dnd_freqs', 1: 'pathfinder_freqs'}
)
ngram_freqs.head()

In [ ]:
plt.figure(figsize = (8,8))
sns.set_palette('dark')
sns.barplot(data = ngram_freqs,
            y = 'ngram',
            x = 'dnd_freqs',
            alpha = 0.5,
            orient = 'h')
sns.barplot(data = ngram_freqs,
            y = 'ngram',
            x = 'pathfinder_freqs',
            alpha = 0.5,
            orient = 'h')
plt.ylabel('n-grams', fontdict = {'fontsize': 12})
plt.xlabel('Number of Posts', fontdict = {'fontsize': 12})
plt.legend(labels = ['r/Pathfinder_RPG', 'r/DungeonsAndDragons'], title = 'Subreddit')
plt.title('Distribution of Frequent n-grams by Subreddit',
          fontdict = {'fontsize': 18});
plt.savefig('../images/ngram_freqs_by_subreddit.png', dpi = 300)

All of the top 20 (1,3)-grams from each subreddit are 1-grams (words). Of these, 10 appear in both subreddits frequently. Given that r/DungeonsAndDragons is less verbose than r/Pathfinder, it makes sense that the words which appear most frequently in in r/DungeonsAndDragons do so far less often than the most frequent words in r/Pathfinder. Regardless, the fact that one-third of the most popular words are shared justifies our decision to employ TFIDF Vectorization when modeling.

### Transform the Data
We have now established that there are certainly characteristics that distinguish r/DungeonsAndDragons posts from r/Pathfinder posts. With the information we have gained from our exploration of the data, we begin modeling. Note that of the factors we explored, one would not otherwise be accounted for by our estimators: compound sentiment. As such, we will include compound sentiment as a feature in our models, which means we will necessarily need to scale our vectorized data.

With that in mind, , there are a couple of steps we must take before fitting any models to the data.

1) Train-Test split. We also select a random seed of 20241126 (the due date of the project) to ensure reproducability
2) Write a general transformer that will convert our data to an amenable form for classification algorithms
3) Establish any assumptions we need

In [ ]:
# Convert all text in the all_text column to lowercase

# Train-test split
X = senti_df[['all_text', 'compound_sentiment']]
y = senti_df['isDnD']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 20241126)

In [ ]:
X.head()

#### Set up a general transformer for the data
To transform our data effectively for modeling, we must take a few steps:

1. Create a lemmatizer to transform each token
2. TFIDF Vectorize the documents with the same stop words we used in our EDA
3. Scale the data by invoking `StandardScaler`(this will only affect some of our classifiers, but we will standarize the process by writing a single transformer which we can feed to all of our models)

We employ some help from ChatGPT to complete this step, as applying `StandardScaler` after `TFIDFVectorizer` can be tricky.

In [ ]:
# These two functions implement POS-aware lemmatization.
# Standard lemmatizers default to treating all tokens as nouns, which produces
# incorrect forms (e.g., "running" → "running" instead of "run").
# We use NLTK's pos_tag to get the part of speech first, then pass it to
# WordNetLemmatizer so each token is lemmatized in the correct grammatical context.


def custom_lemmatize(word: str, tag: str) -> str:
    """
    Lemmatize a single token using its POS tag for context.

    Maps Penn Treebank POS tags to WordNet POS constants and lemmatizes
    accordingly. Words whose POS tag has no WordNet equivalent (e.g., pronouns,
    determiners) are returned unchanged.

    Args:
        word: A single token string.
        tag: Penn Treebank POS tag for the token (e.g., 'NN', 'VBZ', 'JJ').

    Returns:
        Lemmatized form of the word, or the original word if the tag has
        no WordNet mapping.

    Example:
        >>> custom_lemmatize('running', 'VBG')
        'run'
        >>> custom_lemmatize('geese', 'NNS')
        'goose'
    """
    lemmatizer = WordNetLemmatizer()
    pos_map = {
        'J': wordnet.ADJ,
        'V': wordnet.VERB,
        'N': wordnet.NOUN,
        'R': wordnet.ADV
    }
    pos = pos_map.get(tag[0])
    return lemmatizer.lemmatize(word, pos) if pos else word


def lemmatize(text: str) -> str:
    """
    Tokenize and lemmatize a document string using POS-aware lemmatization.

    Splits the text on whitespace, tags each token with its part of speech,
    and applies custom_lemmatize to each token. Designed for use as the
    `preprocessor` argument to TfidfVectorizer.

    Args:
        text: Raw document string (not pre-tokenized).

    Returns:
        A single string of POS-lemmatized tokens joined by spaces.

    Example:
        >>> lemmatize("The wizards are casting spells")
        'The wizard be cast spell'
    """
    tokens = text.split(' ')
    lemmatized_tokens = [custom_lemmatize(word, tag) for word, tag in pos_tag(tokens)]
    return ' '.join(lemmatized_tokens)

In [ ]:
# Create a general transformation pipeline to facilitate fitting the models
# First write a ColumnTransformer to TFIDF Vectorize the text data.
text_preprocessor = ColumnTransformer([
    ('tfidf',
     TfidfVectorizer(stop_words = stop_words,
                     preprocessor = lemmatize,
                     min_df = 0.0025),
     'all_text')
])

# StandardScaler() requires a dense array, so we use FunctionTransformer to make our vectorized data + compound sentiment dense
preprocessor = Pipeline([
    ('tvec', text_preprocessor),
    ('to_dense',
     FunctionTransformer(lambda x: x.toarray(),
                         accept_sparse = True)),
    ('sc', StandardScaler())
])

In [ ]:
processed_train = preprocessor.fit_transform(X_train)
processed_test = preprocessor.transform(X_test)

In [ ]:
processed_train.shape, processed_test.shape

Our data is now uniformly processed and ready to be fed into our models for fitting. Let us first establish our baseline using a Logistic Classifier, the simplest of the classification models in our suite.

### Baseline Model: Logistic Classification
Since we have over 3,000 features post-processing, we will include regularization, even for our baseline. To minimize the impact of an excessive number of features, we employ $L^1$ regularization on our LogisticRegression. For the baseline, we maintain the default value for regularization strength.

In [ ]:
%%time
baseline = LogisticRegression(penalty = 'l1',
                            solver = 'saga',
                            max_iter = 4000,
                            n_jobs = 8,
                            random_state = 20241126)
baseline.fit(processed_train, y_train)

Our baseline logistic classifier is heavily overfit to the data: it scores
99.9% on training but only 80.4% on testing. The gap of nearly 20 percentage
points signals that the model memorized training patterns rather than learning
generalizable ones. We have a long way to go, but the 80.4% test accuracy does
establish a meaningful floor to beat.

In [ ]:
baseline.score(processed_train, y_train), baseline.score(processed_test, y_test)

In [ ]:
baseline_preds = baseline.predict(processed_test)

In [ ]:
print(classification_report(y_test, baseline_preds))

In [ ]:
fig, ax = plt.subplots(figsize = (8,8))
ConfusionMatrixDisplay.from_estimator(baseline,
                                      processed_test,
                                      y_test,
                                      cmap = 'PuBu',
                                      ax = ax)
plt.title('Baseline Confusion Matrix', fontdict = {'fontsize': 18})
plt.savefig('../images/baseline_confusion.png', dpi = 300);

We see that our baseline is consistent across all metrics for classification we've considered in our report. That leads us to believe that on any new data, our baseline will predict the subreddit from which a post originates with 80% accuracy and is just as precise.

### Gridsearch Logistic Regression
Let's considering searching over our hyperparameters to build a better model. We have a few parameters to play with here.

In [ ]:
# Instantiate a new LogisticRegression object to be tuned
logreg = LogisticRegression(solver = 'saga',
                            max_iter = 4000,
                            n_jobs = 12,
                            random_state = 20241126)

# Hyperparameter grid
hyperparams = {'penalty': ['l1', 'l2'],
               'C': np.linspace(start = 0.1, stop = 0.9, num = 5)}

# Instantiate a GridSearchCV object
logreg_gs = GridSearchCV(estimator = logreg,
                  param_grid = hyperparams,
                  n_jobs = 12)

In [ ]:
# Fit the GridSearch over the specified hyperparameters and perform 5-fold cross-validation
logreg_gs.fit(processed_train, y_train)

In [ ]:
# Best estimator
logreg_gs.best_estimator_

In [ ]:
logreg_gs_preds = logreg_gs.best_estimator_.predict(processed_test)

In [ ]:
logreg_gs.best_estimator_.score(processed_train, y_train), logreg_gs.best_estimator_.score(processed_test, y_test)

In [ ]:
print(classification_report(y_test, logreg_gs_preds))

In [ ]:
fig, ax = plt.subplots(figsize = (8,8))
ConfusionMatrixDisplay.from_estimator(logreg_gs.best_estimator_,
                                      processed_test,
                                      y_test,
                                      cmap = 'PuBu',
                                      ax = ax)
plt.title('Optimized Logistic Regression\nConfusion Matrix', fontdict = {'fontsize': 18})
plt.savefig('../images/logreg_confusion.png', dpi = 300);

### K-Nearest Neighbors Classification

In [ ]:
# Instantiate a KNearestClassifier
knn = KNeighborsClassifier()

# Hyperparameter grid
hyperparams = {'n_neighbors': np.arange(3, 30, 2)}

# Instantiate a GridSearchCV object
knn_gs = GridSearchCV(estimator = knn,
                  param_grid = hyperparams,
                  n_jobs = 12)

In [ ]:
# Fit the GridSearch over the specified hyperparameters and perform 5-fold cross-validation
knn_gs.fit(processed_train, y_train)

In [ ]:
# Best estimator
knn_gs.best_estimator_

In [ ]:
knn_gs_preds = knn_gs.best_estimator_.predict(processed_test)

In [ ]:
knn_gs.best_estimator_.score(processed_train, y_train), knn_gs.best_estimator_.score(processed_test, y_test)

In [ ]:
print(classification_report(y_test, knn_gs_preds))

In [ ]:
fig, ax = plt.subplots(figsize = (8,8))
ConfusionMatrixDisplay.from_estimator(knn_gs.best_estimator_,
                                      processed_test,
                                      y_test,
                                      cmap = 'PuBu',
                                      ax = ax)
plt.title('K-Nearest Neighbors\nConfusion Matrix', fontdict = {'fontsize': 18})
plt.savefig('../images/knn_confusion.png', dpi = 300);

### Random Forest Classification

In [ ]:
# Instantiate a RandomForestClassifier
rf = RandomForestClassifier(n_jobs = 12,
                            random_state = 20241126)

# Hyperparameter grid
hyperparams = {'max_depth': np.arange(30, 101, 5),
               'min_samples_split': np.arange(2, 6),
               'max_features': ['sqrt', 'log2']}

# Instantiate a GridSearchCV object
rf_gs = GridSearchCV(estimator = rf,
                  param_grid = hyperparams,
                  n_jobs = 12)

In [ ]:
# Fit the GridSearch over the specified hyperparameters and perform 5-fold cross-validation
rf_gs.fit(processed_train, y_train)

In [ ]:
# Best estimator
rf_gs.best_estimator_

In [ ]:
rf_gs_preds = rf_gs.best_estimator_.predict(processed_test)

In [ ]:
rf_gs.best_estimator_.score(processed_train, y_train), rf_gs.best_estimator_.score(processed_test, y_test)

In [ ]:
print(classification_report(y_test, rf_gs_preds))

In [ ]:
fig, ax = plt.subplots(figsize = (8,8))
ConfusionMatrixDisplay.from_estimator(rf_gs.best_estimator_,
                                      processed_test,
                                      y_test,
                                      cmap = 'PuBu',
                                      ax = ax)
plt.title('Random Forest Confusion Matrix', fontdict = {'fontsize': 18})
plt.savefig('../images/rf_confusion.png', dpi = 300);

### Support Vector Classification

In [ ]:
# Instantiate a Support Vector Classifier
svc = SVC(random_state = 20241126)

# Hyperparameter grid
hyperparams = {'C': np.linspace(start = 0.01, stop = 0.9, num = 10)}

# Instantiate a GridSearchCV object
svc_gs = GridSearchCV(estimator = svc,
                  param_grid = hyperparams,
                  n_jobs = 12)

In [ ]:
# Fit the GridSearch over the specified hyperparameters and perform 5-fold cross-validation
svc_gs.fit(processed_train, y_train)

In [ ]:
# Best estimator
svc_gs.best_estimator_

In [ ]:
svc_gs_preds = svc_gs.best_estimator_.predict(processed_test)

In [ ]:
svc_gs.best_estimator_.score(processed_train, y_train), svc_gs.best_estimator_.score(processed_test, y_test)

In [ ]:
print(classification_report(y_test, svc_gs_preds))

In [ ]:
fig, ax = plt.subplots(figsize = (8,8))
ConfusionMatrixDisplay.from_estimator(svc_gs.best_estimator_,
                                      processed_test,
                                      y_test,
                                      cmap = 'PuBu',
                                      ax = ax)
plt.title('Support Vector Machine\nConfusion Matrix', fontdict = {'fontsize': 18})
plt.savefig('../images/svm_confusion.png', dpi = 300);

### Planned Next Steps

The following analyses were scoped for this project but not completed before the submission deadline:

- **Model evaluation visualization** — Plot GridSearch CV scores across hyperparameter values to show how each model responded to tuning
- **Hyperparameter search visualization** — Surface or line plots comparing regularization strength vs. accuracy for Logistic Regression and SVC
- **ROC Curve / AUC** — Compare all four models on a single ROC plot to give a more complete picture of classifier performance beyond accuracy

### Findings and Implications
Of the four classes of models we tested, two outperformed the baseline (if only just) and two were far outclassed by the baseline. All of our models were fed the same preprocessed data, the pipeline for which was as follows:
1) Employ TFIDF Vectorization to lemmatize and tokenize, remove all (English) stop words, and consider only those words which appear in at most 0.25% of our corpus.
2) Convert the output from a sparse matrix to a dense matrix.
3) Scale the data so that our compound sentiment score is not washed out.

Our baseline logistic regression was able to correctly sort 80% of the testing data. After optimizing the models, the new logistic regression and the random forest classifiers outperformed the baseline slightly, yielding accuracies of 83% each. The k-nearest neighbors and support vector machine classifiers did not bode nearly as well, with each performing below 55% accuracy. Since our models were not able agree on whether the two subreddits are sufficiently distinct, we must concede that the investigation is inconclusive and further analysis is required.

### Next Steps
The models were all overfit, despite our attempt to restrict the sheer number of features. Further analysis would reduce the complexity of the models by further restricting the number of features, increasing the strength of regularization, or fitting less complex models. We could also reduce the scope of our corpus to more verbose documents (those containing body text).